# 04 — CNN passive-PAD proxy experiments & threshold lock

Notebook tự chứa data generation, CNN training và metric. So sánh `MobileNetV3-Small` với `ResNet-18`, chỉ dùng train/validation subjects. Mỗi ảnh tạo một bona-fide sample và một synthetic print/replay/recapture proxy; subject holdout vẫn khóa.

Đây là **PAD proxy**, không phải CelebA-Spoof/Replay-Attack thật và không được dùng để tuyên bố khả năng chống presentation attack ngoài đời. Experiment giữ cùng serving contract của project: ảnh `224×224`, logits 2 class (`0=attack`, `1=bona_fide`) và threshold tại target APCER 5%.


In [1]:
import os
import sys
import json
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn>=1.5", "pandas>=2.2", "seaborn>=0.13",
    "opencv-python-headless>=4.10", "tqdm>=4.66",
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter

SEED = 464
random.seed(SEED)
np.random.seed(SEED)

# A blank Colab runtime is the reference environment.  Nothing from src/ or
# scripts/ is needed.  All intermediate DS assets live under this runtime root.
RUNTIME_ROOT = Path("/content/facekyc_ds") if Path("/content").exists() else Path.cwd() / ".facekyc_ds"
DATA_ROOT = RUNTIME_ROOT / "data"
REPORT_ROOT = RUNTIME_ROOT / "reports"
ARTIFACT_ROOT = RUNTIME_ROOT / "artifacts"
for directory in (DATA_ROOT, REPORT_ROOT, ARTIFACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("Runtime root:", RUNTIME_ROOT)
print("Python:", sys.version.split()[0])


Runtime root: /content/facekyc_ds
Python: 3.13.15


In [2]:
from sklearn.datasets import fetch_lfw_people

# This call downloads the official funneled LFW archive plus pair protocols.
_ = fetch_lfw_people(
    data_home=str(DATA_ROOT), color=True, resize=0.5,
    min_faces_per_person=0, download_if_missing=True,
)
LFW_HOME = DATA_ROOT / "lfw_home"
LFW_IMAGE_ROOT = LFW_HOME / "lfw_funneled"
assert LFW_IMAGE_ROOT.exists(), f"LFW extraction failed: {LFW_IMAGE_ROOT}"

def subject_bucket(subject):
    # Stable subject-level split: 80% train, 10% validation, 10% locked holdout.
    return int(hashlib.sha256(subject.encode("utf-8")).hexdigest()[:8], 16) % 10

def build_lfw_manifest():
    rows = []
    for path in sorted(LFW_IMAGE_ROOT.glob("*/*.jpg")):
        subject = path.parent.name
        bucket = subject_bucket(subject)
        split = "train" if bucket < 8 else ("validation" if bucket == 8 else "holdout")
        rows.append({"path": str(path), "subject": subject, "split": split})
    frame = pd.DataFrame(rows)
    assert len(frame) == 13233, f"Expected 13,233 LFW images, found {len(frame)}"
    assert frame.groupby("subject")["split"].nunique().max() == 1
    return frame

MANIFEST_PATH = DATA_ROOT / "lfw_subject_manifest.csv"
manifest = build_lfw_manifest()
manifest.to_csv(MANIFEST_PATH, index=False)
print("LFW images:", len(manifest), "subjects:", manifest.subject.nunique())


LFW images: 13233 subjects: 5749


In [3]:
from sklearn.metrics import roc_auc_score

def verification_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    accepted = scores >= threshold
    impostor = labels == 0
    genuine = labels == 1
    return {
        "threshold": float(threshold),
        "fmr": float(accepted[impostor].mean()),
        "fnmr": float((~accepted[genuine]).mean()),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, scores)),
        "pairs": int(len(labels)),
    }

def threshold_for_fmr(labels, scores, target_fmr=0.01):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    impostor_scores = np.sort(scores[labels == 0])
    rank = int(np.ceil((1.0 - target_fmr) * len(impostor_scores))) - 1
    rank = int(np.clip(rank, 0, len(impostor_scores) - 1))
    return float(np.nextafter(impostor_scores[rank], np.inf))

def pad_metrics(labels, live_scores, threshold):
    labels = np.asarray(labels).astype(int)  # 0=attack, 1=bona fide
    live_scores = np.asarray(live_scores, dtype=float)
    accepted = live_scores >= threshold
    attack = labels == 0
    bona_fide = labels == 1
    apcer = float(accepted[attack].mean())
    bpcer = float((~accepted[bona_fide]).mean())
    return {
        "threshold": float(threshold), "apcer": apcer, "bpcer": bpcer,
        "acer": float((apcer + bpcer) / 2),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, live_scores)),
        "samples": int(len(labels)),
    }

def threshold_for_apcer(labels, live_scores, target_apcer=0.05):
    labels = np.asarray(labels).astype(int)
    attack_scores = np.sort(np.asarray(live_scores, dtype=float)[labels == 0])
    rank = int(np.ceil((1.0 - target_apcer) * len(attack_scores))) - 1
    rank = int(np.clip(rank, 0, len(attack_scores) - 1))
    return float(np.nextafter(attack_scores[rank], np.inf))


In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

INPUT_SIZE = 224
TARGET_APCER = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def synthetic_presentation_attack(image, seed):
    """Deterministic print/replay proxy; not a real PAD capture."""
    rng = np.random.default_rng(seed)
    image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR).convert("RGB")
    array = np.asarray(image).astype(np.float32)
    attack_type = int(seed % 3)
    if attack_type == 0:  # print: paper tint, reduced gamut, dot/noise texture
        gray = array.mean(axis=2, keepdims=True)
        array = 0.65 * array + 0.35 * gray
        array *= np.array([1.04, 1.00, 0.90], dtype=np.float32)
        array += rng.normal(0, 7, array.shape)
        array[::4, ::4] *= 0.82
    elif attack_type == 1:  # replay: scanlines, blue cast and screen glare
        array *= np.array([0.92, 0.98, 1.10], dtype=np.float32)
        array[::3] *= 0.72
        x = np.linspace(-1, 1, array.shape[1])[None, :, None]
        glare = 34 * np.exp(-((x - 0.35) ** 2) / 0.05)
        array += glare
    else:  # recapture: resampling, compression-like blocking and moire
        small = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8)).resize((64, 64))
        array = np.asarray(small.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR)).astype(np.float32)
        yy, xx = np.indices(array.shape[:2])
        moire = 12 * np.sin((xx + yy) * 0.42)[..., None]
        array += moire
    return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))

class PADProxyDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        augment = [transforms.Resize((INPUT_SIZE, INPUT_SIZE))]
        if training:
            augment += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.12, 0.12, 0.08, 0.02)]
        augment += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
        self.transform = transforms.Compose(augment)

    def __len__(self):
        return len(self.frame) * 2

    def __getitem__(self, index):
        base_index, label = divmod(index, 2)  # 0=attack, 1=bona fide
        row = self.frame.iloc[base_index]
        with Image.open(row.path) as source:
            image = source.convert("RGB")
        if label == 0:
            seed = int(hashlib.sha256(row.path.encode()).hexdigest()[:8], 16)
            image = synthetic_presentation_attack(image, seed)
        return self.transform(image), torch.tensor(label, dtype=torch.long), str(row.subject)

def build_pad_cnn(name):
    if name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        for parameter in model.features.parameters():
            parameter.requires_grad = False
        for parameter in model.features[-2:].parameters():
            parameter.requires_grad = True
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    elif name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.layer4.parameters():
            parameter.requires_grad = True
        model.fc = nn.Linear(model.fc.in_features, 2)
    else:
        raise ValueError(name)
    return model.to(DEVICE)

def evaluate_pad(model, loader):
    model.eval()
    scores, labels = [], []
    with torch.inference_mode():
        for images, target, _ in loader:
            probabilities = torch.softmax(model(images.to(DEVICE, non_blocking=True)), dim=1)[:, 1]
            scores.extend(probabilities.cpu().numpy().tolist())
            labels.extend(target.numpy().astype(int).tolist())
    return np.asarray(labels), np.asarray(scores)

def train_pad_candidate(name, train_loader, validation_loader, epochs=3):
    model = build_pad_cnn(name)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()
    history = []
    best = None
    for epoch in range(1, epochs + 1):
        model.train()
        running = []
        for images, target, _ in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), target)
            loss.backward()
            optimizer.step()
            running.append(float(loss.detach().cpu()))
        labels, scores = evaluate_pad(model, validation_loader)
        threshold = threshold_for_apcer(labels, scores, target_apcer=TARGET_APCER)
        metrics = pad_metrics(labels, scores, threshold)
        history.append({"epoch": epoch, "train_loss": float(np.mean(running)), **metrics})
        candidate = (metrics["acer"], metrics["bpcer"], epoch)
        if best is None or candidate < best[0]:
            best = (candidate, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, threshold, metrics, epoch)
        print(name, history[-1])
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best[1], best[2], best[3], history, best[4]


Device: cuda Tesla T4


In [5]:
assert DEVICE.type == "cuda", "Notebook 04 cần Colab GPU."
train_frame = manifest.query("split == 'train'").sample(n=1600, random_state=SEED)
validation_frame = manifest.query("split == 'validation'").sample(
    n=min(400, len(manifest.query("split == 'validation'"))), random_state=SEED
)
assert not (set(train_frame.subject) & set(validation_frame.subject))

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(PADProxyDataset(train_frame, training=True), batch_size=64,
                          shuffle=True, num_workers=2, pin_memory=DEVICE.type == "cuda",
                          generator=loader_generator)
validation_loader = DataLoader(PADProxyDataset(validation_frame, training=False), batch_size=128,
                               shuffle=False, num_workers=2, pin_memory=DEVICE.type == "cuda")
print("Train samples:", len(train_loader.dataset), "validation samples:", len(validation_loader.dataset))


Train samples: 3200 validation samples: 800


In [6]:
candidates = []
trained_state_dicts = {}
for architecture in ["mobilenet_v3_small", "resnet18"]:
    # Give both candidates the same shuffle/augmentation seed for a fairer comparison.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)
    loader_generator.manual_seed(SEED)
    state_dict, threshold, metrics, history, best_epoch = train_pad_candidate(
        architecture, train_loader, validation_loader, epochs=3
    )
    candidates.append({"architecture": architecture, "threshold": threshold,
                       "best_epoch": best_epoch, "validation_metrics": metrics,
                       "history": history})
    trained_state_dicts[architecture] = state_dict

scorecard = pd.DataFrame([
    {"architecture": row["architecture"], **row["validation_metrics"]}
    for row in candidates
]).sort_values(["acer", "bpcer"])
display(scorecard)
selected = str(scorecard.iloc[0].architecture)
selected_record = next(row for row in candidates if row["architecture"] == selected)
print("Selected PAD CNN:", selected)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 118MB/s]


mobilenet_v3_small {'epoch': 1, 'train_loss': 0.11294298933120445, 'threshold': 0.9739118814468385, 'apcer': 0.05, 'bpcer': 0.0325, 'acer': 0.04125, 'accuracy': 0.95875, 'auc': 0.9922625, 'samples': 800}
mobilenet_v3_small {'epoch': 2, 'train_loss': 0.007056140307104215, 'threshold': 0.7619086503982545, 'apcer': 0.05, 'bpcer': 0.0, 'acer': 0.025, 'accuracy': 0.975, 'auc': 0.99908125, 'samples': 800}
mobilenet_v3_small {'epoch': 3, 'train_loss': 0.004820930502610281, 'threshold': 0.1882279068231583, 'apcer': 0.05, 'bpcer': 0.0, 'acer': 0.025, 'accuracy': 0.975, 'auc': 0.99993125, 'samples': 800}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 178MB/s]


resnet18 {'epoch': 1, 'train_loss': 0.025436047703624353, 'threshold': 0.0006767725571990014, 'apcer': 0.05, 'bpcer': 0.0, 'acer': 0.025, 'accuracy': 0.975, 'auc': 1.0, 'samples': 800}
resnet18 {'epoch': 2, 'train_loss': 0.0006542365733184851, 'threshold': 0.00011391602311050521, 'apcer': 0.05, 'bpcer': 0.0, 'acer': 0.025, 'accuracy': 0.975, 'auc': 1.0, 'samples': 800}
resnet18 {'epoch': 3, 'train_loss': 0.0017258243116611994, 'threshold': 0.00018615972658153626, 'apcer': 0.05, 'bpcer': 0.0, 'acer': 0.025, 'accuracy': 0.975, 'auc': 1.0, 'samples': 800}


,architecture,threshold,apcer,bpcer,acer,accuracy,auc,samples
0,mobilenet_v3_small,0.761909,0.05,0.0,0.025,0.975,0.999081,800
1,resnet18,0.000677,0.05,0.0,0.025,0.975,1.000000,800


Selected PAD CNN: mobilenet_v3_small


In [7]:
checkpoint_path = ARTIFACT_ROOT / "pad_proxy_selected.pt"
torch.save({
    "architecture": selected,
    "state_dict": trained_state_dicts[selected],
    "threshold": selected_record["threshold"],
    "input_size": INPUT_SIZE,
    "class_contract": {"0": "attack", "1": "bona_fide"},
    "score": "softmax_probability_class_1",
}, checkpoint_path)
checkpoint_digest = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()

report = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "stage": "validation_model_selection_and_threshold_lock",
    "dataset": "subject-disjoint LFW-derived PAD proxy",
    "seed": SEED,
    "subject_disjoint": True,
    "input_size": INPUT_SIZE,
    "score": "softmax_probability_class_1",
    "base_train_images": int(len(train_frame)),
    "base_validation_images": int(len(validation_frame)),
    "candidate_backbones": ["mobilenet_v3_small", "resnet18"],
    "target_apcer": TARGET_APCER,
    "selected_architecture": selected,
    "best_epoch": selected_record["best_epoch"],
    "locked_threshold": selected_record["threshold"],
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": checkpoint_digest,
    "validation_metrics": selected_record["validation_metrics"],
    "scorecard": candidates,
    "holdout_accessed": False,
    "production_ready": False,
    "selection_note": "Architecture, epoch and threshold use the same validation split; metrics are for selection/calibration, not an unbiased final estimate.",
    "limitation": "Synthetic texture attacks do not establish real passive-PAD performance.",
}
output = REPORT_ROOT / "04_pad_selection.json"
output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
report


{'created_at': '2026-08-31T06:08:23.015513+00:00',
 'stage': 'validation_model_selection_and_threshold_lock',
 'dataset': 'subject-disjoint LFW-derived PAD proxy',
 'seed': 464,
 'subject_disjoint': True,
 'input_size': 224,
 'score': 'softmax_probability_class_1',
 'base_train_images': 1600,
 'base_validation_images': 400,
 'candidate_backbones': ['mobilenet_v3_small', 'resnet18'],
 'target_apcer': 0.05,
 'selected_architecture': 'mobilenet_v3_small',
 'best_epoch': 2,
 'locked_threshold': 0.7619086503982545,
 'checkpoint_path': '/content/facekyc_ds/artifacts/pad_proxy_selected.pt',
 'checkpoint_sha256': '490a9c962d6b2262895a739a9507a000850fe6771c46cb01f54d0b99cb03d492',
 'validation_metrics': {'threshold': 0.7619086503982545,
  'apcer': 0.05,
  'bpcer': 0.0,
  'acer': 0.025,
  'accuracy': 0.975,
  'auc': 0.99908125,
  'samples': 800},
 'scorecard': [{'architecture': 'mobilenet_v3_small',
   'threshold': 0.7619086503982545,
   'best_epoch': 2,
   'validation_metrics': {'threshold': 0.

## Kết luận sau khi chạy

Model/epoch/threshold chỉ được chọn từ validation, nên metric ở đây là metric phục vụ selection/calibration chứ chưa phải ước lượng cuối độc lập. Synthetic proxy kiểm tra được pipeline, contract 2 class và APCER/BPCER implementation, nhưng dễ học scanline/noise shortcuts. Chỉ notebook 05 mới được mở subject holdout. Trạng thái deployment phải giữ `candidate` cho tới khi đánh giá subject/device/attack-disjoint trên PAD dataset thật.
